In [3]:
import numpy as np
import matplotlib.pyplot as plt
import math

In [5]:
from sklearn.utils import shuffle

# preluam datele de antrenare
training_data = np.load('data_lab6/data/training_data.npy')
prices = np.load('data_lab6/data/prices.npy')

print('The first 4 samples are:\n ', training_data[:4])
print('The first 4 prices are:\n ', prices[:4])

# shuffle
training_data, prices = shuffle(training_data, prices, random_state=0)

The first 4 samples are:
  [[2.0150e+03 4.1000e+04 1.9670e+01 1.5820e+03 1.2620e+02 5.0000e+00
  1.0000e+00 0.0000e+00 1.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00
  1.0000e+00 0.0000e+00]
 [2.0110e+03 4.6000e+04 1.8200e+01 1.1990e+03 8.8700e+01 5.0000e+00
  1.0000e+00 0.0000e+00 0.0000e+00 1.0000e+00 0.0000e+00 0.0000e+00
  1.0000e+00 0.0000e+00]
 [2.0120e+03 8.7000e+04 2.0770e+01 1.2480e+03 8.8760e+01 7.0000e+00
  1.0000e+00 0.0000e+00 1.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00
  1.0000e+00 0.0000e+00]
 [2.0130e+03 8.6999e+04 2.3080e+01 1.4610e+03 6.3100e+01 5.0000e+00
  1.0000e+00 0.0000e+00 1.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00
  1.0000e+00 0.0000e+00]]
The first 4 prices are:
  [12.5  4.5  6.   3.5]


In [7]:
from sklearn import preprocessing
def normalize_data(train_data, test_data):
    scaler = preprocessing.StandardScaler()
    scaler.fit(train_data)
    scaled_train_data = scaler.transform(train_data)
    scaled_test_data = scaler.transform(test_data)
    return scaled_train_data, scaled_test_data

In [9]:
print(training_data.shape)
print(prices.shape)

(4879, 14)
(4879,)


In [11]:
def generate_folds(all_data, k):
    fold_length = math.floor(np.size(all_data,0) / k)
    print(fold_length)
    indices = np.arange(len(all_data))
    # folds - o lista de tupluri de forma (train_indices, test_indices)
    folds = list()
    for idx in range(k):
        # indices = shuffle(indices)
        test_indices = np.array(indices[idx * fold_length : (idx+1) * fold_length])
        train_indices = np.concatenate([indices[:idx * fold_length], indices[(idx + 1) * fold_length:]])
        folds.append((train_indices, test_indices))
        print(f'Indici train: {train_indices}')
    return folds
        
F = generate_folds(training_data, 3)
print(F)

1626
Indici train: [1626 1627 1628 ... 4876 4877 4878]
Indici train: [   0    1    2 ... 4876 4877 4878]
Indici train: [   0    1    2 ... 3250 3251 4878]
[(array([1626, 1627, 1628, ..., 4876, 4877, 4878]), array([   0,    1,    2, ..., 1623, 1624, 1625])), (array([   0,    1,    2, ..., 4876, 4877, 4878]), array([1626, 1627, 1628, ..., 3249, 3250, 3251])), (array([   0,    1,    2, ..., 3250, 3251, 4878]), array([3252, 3253, 3254, ..., 4875, 4876, 4877]))]


In [15]:
# antrenarea modelului de regresie liniara, folosind validarea incrucisata cu 3 fold-uri

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error

linear_regression_model = LinearRegression()
mse_scores, mae_scores = [], []

for train_indices, test_indices in F:
    X_train, y_train = training_data[train_indices], prices[train_indices]
    X_test, y_true = training_data[test_indices], prices[test_indices]

    print(X_train.shape)
    print(X_test.shape)
    
    scaled_train_data, scaled_test_data = normalize_data(X_train, X_test)
    linear_regression_model.fit(scaled_train_data, y_train)

    y_pred = linear_regression_model.predict(scaled_test_data)

    mae_value = mean_absolute_error(y_true, y_pred)
    mse_value = mean_squared_error(y_true, y_pred)

    mse_scores.append(mse_value)
    mae_scores.append(mae_value)

mean_mse = sum(mse_scores) / len(mse_scores)
mean_mae = sum(mae_scores) / len(mae_scores)

print(f'Media MSE: {round(mean_mse, 2)}')
print(f'Media MAE: {round(mean_mae, 2)}')

(3253, 14)
(1626, 14)
(3253, 14)
(1626, 14)
(3253, 14)
(1626, 14)
Media MSE: 3.17
Media MAE: 1.32


In [17]:
alphas = np.array([1, 10, 100, 1000])

best_score = float("inf")

for a in alphas:
    ridge_regression_model = Ridge(alpha=a)
    mse_scores, mae_scores = [], []

    for train_indices, test_indices in F:
        X_train, y_train = training_data[train_indices], prices[train_indices]
        X_test, y_true = training_data[test_indices], prices[test_indices]
        
        scaled_train_data, scaled_test_data = normalize_data(X_train, X_test)
        ridge_regression_model.fit(scaled_train_data, y_train)
    
        y_pred = ridge_regression_model.predict(scaled_test_data)
    
        mae_value = mean_absolute_error(y_true, y_pred)
        mse_value = mean_squared_error(y_true, y_pred)
    
        mse_scores.append(mse_value)
        mae_scores.append(mae_value)

    mean_mse = sum(mse_scores) / len(mse_scores)
    mean_mae = sum(mae_scores) / len(mae_scores)
    
    if mean_mse < best_score:
        best_score = mean_mse
        best_alpha = a
        
    print(f'Alpha:{a}')
    print(f'Media MSE: {round(mean_mse, 5)}')
    print(f'Media MAE: {round(mean_mae, 5)}\n')
    
print(f'Alpha cu cea mai buna performanta: {best_alpha}')

Alpha:1
Media MSE: 3.16653
Media MAE: 1.31923

Alpha:10
Media MSE: 3.16637
Media MAE: 1.31902

Alpha:100
Media MSE: 3.17122
Media MAE: 1.31821

Alpha:1000
Media MSE: 3.43175
Media MAE: 1.36613

Alpha cu cea mai buna performanta: 10


In [21]:

rdg_model = Ridge(alpha = best_alpha)

X_train, y_train = training_data, prices
X_test, y_true = training_data[:math.floor(np.size(training_data, 0) / 3)], prices[:math.floor(np.size(training_data, 0) / 3)]
scaled_train_data, scaled_test_data = normalize_data(X_train, X_test)

rdg_model.fit(scaled_train_data, y_train)
    
y_pred = rdg_model.predict(scaled_test_data)



In [47]:
nume_features = {1:"anul fabricatiei", 2:"numarul de kilometri", 3:"mileage", 4:"motor", 5:"putere", 6:"numar locuri",
                7:"numar proprietari", 8:"combustibil1", 9:"combustibil2", 10:"combustibil3", 11:"combustibil4", 12:"combustibil5",
                13:"transmisia manuala", 14:"transmisia automata"}
print(np.array(rdg_model.coef_))
coeficienti = np.argsort(rdg_model.coef_)

bias = rdg_model.intercept_

coeficienti += 1
print(coeficienti)
print(f'Cel mai semnificativ atribut: {nume_features[coeficienti[-1]]}')
print(f'Al doilea cel mai semnificativ atribut: {nume_features[coeficienti[-2]]}')
print(f'Cel mai putin semnificativ atribut: {nume_features[coeficienti[0]]}')
print(f'Bias-ul: {bias}')


[ 1.6635199  -0.15533425 -0.46034902  0.40463108  1.3356845   0.13251059
 -0.08683589  0.          0.36666477 -0.3666661   0.          0.
 -0.2293787   0.2293682 ]
[ 3 10 13  2  7  8 11 12  6 14  9  4  5  1]
Cel mai semnificativ atribut: anul fabricatiei
Al doilea cel mai semnificativ atribut: putere
Cel mai putin semnificativ atribut: mileage
Bias-ul: 5.695129871368408
